# Classical Machine Learning on Pretrained Feature Extractors

This report presents the findings of applying classical Machine Learning models (Logistic Regression and Linear SVM) on features extracted from various deep neural network backbones.
We explore backbones such as **ConvNeXt-Tiny**, **ResNet** variants, and **EfficientNet** variants.
The dataset used is `mini-imagenet`.

## 1. Setup and Initialization
Import necessary modules and load the dataset. We have moved out the utility and model definition functions into `utils.py` and `approach1_utils.py` to keep this notebook tidy.

In [1]:
import os
import json
from pathlib import Path
import torch
import pprint

from utils import set_seed, get_device, load_mini_imagenet, make_transforms
from approach1_utils import classical_ml_experiment

seed = 24
set_seed(seed)
use_gpu = torch.cuda.is_available() or torch.backends.mps.is_available()
device = get_device(use_gpu)
print(f"Using device: {device}")

# Load the dataset
ds = load_mini_imagenet(subset=None)
save_dir = Path("experiments/classical_ml")
save_dir.mkdir(parents=True, exist_ok=True)


/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


Generating test split: 100%|██████████| 5000/5000 [00:00<00:00, 24093.70 examples/s]


## 2. Backbone Architectures
Before presenting the experiments, we briefly discuss the deep neural network architectures used as feature extractors in this report.

### ConvNeXt (e.g., `convnext_tiny`)
**Description:** Introduced in 2022, ConvNeXt is a modernized, pure convolutional neural network that competes directly with Vision Transformers (ViTs). It incorporates several design choices from ViTs into standard ResNet paradigms, including larger kernel sizes (7x7), inverted bottlenecks, depthwise convolutions, substituting BatchNorm with LayerNorm, and replacing ReLU with GELU. The `convnext_tiny` variant provides a highly efficient yet powerful baseline.

**Schematic Representation:**
```text
Input Image
    │
    ▼
[ Stem: 4x4 Conv, Stride 4 + LayerNorm ]   <-- Patchify
    │
    ▼
┌───────────────────────────────────────┐
│          ConvNeXt Block (x N)         │
│  1. Depthwise Conv (7x7)              │
│  2. LayerNorm                         │
│  3. Pointwise Conv (1x1) + GELU       │
│  4. Pointwise Conv (1x1)              │
│  (+ Residual Skip Connection)         │
└───────────────────────────────────────┘
    │
    ▼
[ Global Average Pooling ]
    │
    ▼
[ Final Classifier Output ]
```

### ResNet (e.g., `resnet18`, `resnet34`, `resnet50`)
**Description:** Residual Networks (ResNets) revolutionized deep learning by introducing skip (residual) connections. This allows for training extremely deep models by mitigating the vanishing gradient problem, enabling the network to learn identity functions if additional layers are not necessary. ResNet-18 and 34 use basic building blocks, while ResNet-50 uses bottleneck blocks for parameter efficiency.

**Schematic Representation:**
```text
Input Image
    │
    ▼
[ Conv1: 7x7 Conv, Stride 2 + BatchNorm + ReLU ]
[ MaxPool: 3x3, Stride 2 ]
    │
    ▼
┌───────────────────────────────────────┐
│        Residual Block (x N)           │
│  1. Conv2d (3x3) + BatchNorm + ReLU   │
│  2. Conv2d (3x3) + BatchNorm          │
│  (+ Residual Skip Connection)         │
│  3. ReLU                              │
└───────────────────────────────────────┘
    │
    ▼
[ Global Average Pooling ]
    │
    ▼
[ Final Classifier Output ]
```

### EfficientNet (e.g., `efficientnet_b0`, `efficientnet_b1`)
**Description:** EfficientNet introduced a systematic "compound scaling" method to scale CNNs appropriately across depth (number of layers), width (number of channels), and resolution (image size) uniformly. Its base block relies on Mobile Inverted Bottleneck Convolutions (MBConv) along with Squeeze-and-Excitation optimization, resulting in state-of-the-art accuracy with fewer parameters and operations.

**Schematic Representation:**
```text
Input Image
    │
    ▼
[ Stem: 3x3 Conv + BatchNorm + Swish ]
    │
    ▼
┌───────────────────────────────────────┐
│          MBConv Block (x N)           │
│  1. Expand (1x1 Conv) + Swish         │
│  2. Depthwise Conv (3x3/5x5) + Swish  │
│  3. Squeeze-and-Excitation            │
│  4. Project (1x1 Conv) + Linear       │
│  (+ Residual Skip Connection)         │
└───────────────────────────────────────┘
    │
    ▼
[ Head: 1x1 Conv + Global Average Pooling ]
    │
    ▼
[ Final Classifier Output ]
```

## 3. Linear SVM Classifiers
In this section, we extract features and train a Linear Support Vector Machine on different backbones. Image size is set to 224x224.

**Linear Support Vector Machine (SVM):**
Support Vector Machines are powerful supervised learning models used for classification. A Linear SVM aims to find the optimal separating hyperplane that maximizes the margin between different classes in the feature space. Because the generic feature representations extracted from our deep neural network layers (such as ConvNeXt, ResNet, and EfficientNet) inhabit a very high-dimensional space and are naturally highly linearly separable, Linear SVMs provide a robust, fast, and mathematically sound classification mechanism without incurring the computational penalty of non-linear kernel tricks.

### SVM with `convnext_tiny`\Extract features using the `convnext_tiny` architecture and fit a Linear SVM.

In [2]:
_, eval_tf, _ = make_transforms(img_size=224)
results = classical_ml_experiment(
    ds=ds, 
    eval_tf=eval_tf, 
    device=device, 
    backbone="convnext_tiny", 
    clf_type="linear_svm", 
    batch_size=256, 
    save_dir=save_dir, 
    img_size=224
)
print("Final Result:")
pprint.pprint(results)

[convnext_tiny | linear_svm @ 224] Extracting features ...


/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))


Saved -> experiments/classical_ml/classical_ml_linear_svm_convnext_tiny_224.json
Final Result:
{'GPU': True,
 'Image Size': 224,
 'Inference Time per Image (ms)': 5.474546098709106,
 'Number of parameters (mil)': 27.897028,
 'Pretrained Models': True,
 'Test accuracy (%)': 93.89999999999999,
 'Test loss (Cross Entropy)': nan,
 'Training Time (seconds)': 131.09030771255493,
 'approach': 'classical_ml',
 'backbone': 'convnext_tiny',
 'classifier': 'linear_svm',
 'test_acc': 0.939,
 'val_acc': 0.9835}


### SVM with `resnet18`\Extract features using the `resnet18` architecture and fit a Linear SVM.

In [3]:
_, eval_tf, _ = make_transforms(img_size=224)
results = classical_ml_experiment(
    ds=ds, 
    eval_tf=eval_tf, 
    device=device, 
    backbone="resnet18", 
    clf_type="linear_svm", 
    batch_size=256, 
    save_dir=save_dir, 
    img_size=224
)
print("Final Result:")
pprint.pprint(results)

[resnet18 | linear_svm @ 224] Extracting features ...


/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))


Saved -> experiments/classical_ml/classical_ml_linear_svm_resnet18_224.json
Final Result:
{'GPU': True,
 'Image Size': 224,
 'Inference Time per Image (ms)': 3.4281486034393307,
 'Number of parameters (mil)': 11.227812,
 'Pretrained Models': True,
 'Test accuracy (%)': 83.26,
 'Test loss (Cross Entropy)': nan,
 'Training Time (seconds)': 267.08700013160706,
 'approach': 'classical_ml',
 'backbone': 'resnet18',
 'classifier': 'linear_svm',
 'test_acc': 0.8326,
 'val_acc': 0.8698}


### SVM with `resnet34`\Extract features using the `resnet34` architecture and fit a Linear SVM.

In [4]:
_, eval_tf, _ = make_transforms(img_size=224)
results = classical_ml_experiment(
    ds=ds, 
    eval_tf=eval_tf, 
    device=device, 
    backbone="resnet34", 
    clf_type="linear_svm", 
    batch_size=256, 
    save_dir=save_dir, 
    img_size=224
)
print("Final Result:")
pprint.pprint(results)

[resnet34 | linear_svm @ 224] Extracting features ...


/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))


Saved -> experiments/classical_ml/classical_ml_linear_svm_resnet34_224.json
Final Result:
{'GPU': True,
 'Image Size': 224,
 'Inference Time per Image (ms)': 3.7641497611999513,
 'Number of parameters (mil)': 21.335972,
 'Pretrained Models': True,
 'Test accuracy (%)': 85.66,
 'Test loss (Cross Entropy)': nan,
 'Training Time (seconds)': 186.98036217689514,
 'approach': 'classical_ml',
 'backbone': 'resnet34',
 'classifier': 'linear_svm',
 'test_acc': 0.8566,
 'val_acc': 0.9135}


### SVM with `resnet50`\Extract features using the `resnet50` architecture and fit a Linear SVM.

In [5]:
_, eval_tf, _ = make_transforms(img_size=224)
results = classical_ml_experiment(
    ds=ds, 
    eval_tf=eval_tf, 
    device=device, 
    backbone="resnet50", 
    clf_type="linear_svm", 
    batch_size=256, 
    save_dir=save_dir, 
    img_size=224
)
print("Final Result:")
pprint.pprint(results)

[resnet50 | linear_svm @ 224] Extracting features ...


/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))


Saved -> experiments/classical_ml/classical_ml_linear_svm_resnet50_224.json
Final Result:
{'GPU': True,
 'Image Size': 224,
 'Inference Time per Image (ms)': 5.049988603591919,
 'Number of parameters (mil)': 23.712932,
 'Pretrained Models': True,
 'Test accuracy (%)': 93.26,
 'Test loss (Cross Entropy)': nan,
 'Training Time (seconds)': 673.6871659755707,
 'approach': 'classical_ml',
 'backbone': 'resnet50',
 'classifier': 'linear_svm',
 'test_acc': 0.9326,
 'val_acc': 0.971}


### SVM with `efficientnet_b0`\Extract features using the `efficientnet_b0` architecture and fit a Linear SVM.

In [6]:
_, eval_tf, _ = make_transforms(img_size=224)
results = classical_ml_experiment(
    ds=ds, 
    eval_tf=eval_tf, 
    device=device, 
    backbone="efficientnet_b0", 
    clf_type="linear_svm", 
    batch_size=256, 
    save_dir=save_dir, 
    img_size=224
)
print("Final Result:")
pprint.pprint(results)

[efficientnet_b0 | linear_svm @ 224] Extracting features ...


/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))


Saved -> experiments/classical_ml/classical_ml_linear_svm_efficientnet_b0_224.json
Final Result:
{'GPU': True,
 'Image Size': 224,
 'Inference Time per Image (ms)': 3.9521808147430417,
 'Number of parameters (mil)': 4.135648,
 'Pretrained Models': True,
 'Test accuracy (%)': 91.94,
 'Test loss (Cross Entropy)': nan,
 'Training Time (seconds)': 1751.9931800365448,
 'approach': 'classical_ml',
 'backbone': 'efficientnet_b0',
 'classifier': 'linear_svm',
 'test_acc': 0.9194,
 'val_acc': 0.9575}


### SVM with `efficientnet_b1`\Extract features using the `efficientnet_b1` architecture and fit a Linear SVM.

In [7]:
_, eval_tf, _ = make_transforms(img_size=224)
results = classical_ml_experiment(
    ds=ds, 
    eval_tf=eval_tf, 
    device=device, 
    backbone="efficientnet_b1", 
    clf_type="linear_svm", 
    batch_size=256, 
    save_dir=save_dir, 
    img_size=224
)
print("Final Result:")
pprint.pprint(results)

[efficientnet_b1 | linear_svm @ 224] Extracting features ...


/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))


Saved -> experiments/classical_ml/classical_ml_linear_svm_efficientnet_b1_224.json
Final Result:
{'GPU': True,
 'Image Size': 224,
 'Inference Time per Image (ms)': 4.569148969650269,
 'Number of parameters (mil)': 6.641284,
 'Pretrained Models': True,
 'Test accuracy (%)': 92.56,
 'Test loss (Cross Entropy)': nan,
 'Training Time (seconds)': 346.1028769016266,
 'approach': 'classical_ml',
 'backbone': 'efficientnet_b1',
 'classifier': 'linear_svm',
 'test_acc': 0.9256,
 'val_acc': 0.9665}


## 4. Logistic Regression Classifiers
In this section, we train Logistic Regression classifiers. To observe the size variation and possibly improve speeds against standard ImageNet sizes, we try an image size of 32x32.

**Logistic Regression (LogReg):**
Logistic regression is a fundamental statistical model that uses the logistic function to model probabilities, gracefully extending to multiclass classification via multinomial distributions. By applying Logistic Regression directly on the extracted backbone features, the model learns the optimal linear combination of deep feature activations to predict the final log-odds for each class. It yields beautifully calibrated class probabilities. We primarily experiment with Logistic Regression alongside a smaller raw image size (`32x32`) to measure drastic trade-offs in speed versus raw validation accuracy.

### Logistic Regression with `convnext_tiny`\Extract features using the `convnext_tiny` architecture scaled to `32x32` and fit Logistic Regression.

In [8]:
_, eval_tf, _ = make_transforms(img_size=32)
results = classical_ml_experiment(
    ds=ds, 
    eval_tf=eval_tf, 
    device=device, 
    backbone="convnext_tiny", 
    clf_type="logreg", 
    batch_size=256, 
    save_dir=save_dir, 
    img_size=32
)
print("Final Result:")
pprint.pprint(results)

[convnext_tiny | logreg @ 32] Extracting features ...


/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))
/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Saved -> experiments/classical_ml/classical_ml_logreg_convnext_tiny_32.json
Final Result:
{'GPU': True,
 'Image Size': 32,
 'Inference Time per Image (ms)': 3.02386155128479,
 'Number of parameters (mil)': 27.897028,
 'Pretrained Models': True,
 'Test accuracy (%)': 44.18,
 'Test loss (Cross Entropy)': 3.6851509845529056,
 'Training Time (seconds)': 22.74753499031067,
 'approach': 'classical_ml',
 'backbone': 'convnext_tiny',
 'classifier': 'logreg',
 'test_acc': 0.4418,
 'val_acc': 0.4578}


### Logistic Regression with `resnet18`\Extract features using the `resnet18` architecture scaled to `32x32` and fit Logistic Regression.

In [9]:
_, eval_tf, _ = make_transforms(img_size=32)
results = classical_ml_experiment(
    ds=ds, 
    eval_tf=eval_tf, 
    device=device, 
    backbone="resnet18", 
    clf_type="logreg", 
    batch_size=256, 
    save_dir=save_dir, 
    img_size=32
)
print("Final Result:")
pprint.pprint(results)

[resnet18 | logreg @ 32] Extracting features ...


/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))
/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Saved -> experiments/classical_ml/classical_ml_logreg_resnet18_32.json
Final Result:
{'GPU': True,
 'Image Size': 32,
 'Inference Time per Image (ms)': 3.1002722740173336,
 'Number of parameters (mil)': 11.227812,
 'Pretrained Models': True,
 'Test accuracy (%)': 32.4,
 'Test loss (Cross Entropy)': 3.111192031727401,
 'Training Time (seconds)': 30.6908278465271,
 'approach': 'classical_ml',
 'backbone': 'resnet18',
 'classifier': 'logreg',
 'test_acc': 0.324,
 'val_acc': 0.341}


### Logistic Regression with `resnet34`\Extract features using the `resnet34` architecture scaled to `32x32` and fit Logistic Regression.

In [10]:
_, eval_tf, _ = make_transforms(img_size=32)
results = classical_ml_experiment(
    ds=ds, 
    eval_tf=eval_tf, 
    device=device, 
    backbone="resnet34", 
    clf_type="logreg", 
    batch_size=256, 
    save_dir=save_dir, 
    img_size=32
)
print("Final Result:")
pprint.pprint(results)

[resnet34 | logreg @ 32] Extracting features ...


/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))
/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Saved -> experiments/classical_ml/classical_ml_logreg_resnet34_32.json
Final Result:
{'GPU': True,
 'Image Size': 32,
 'Inference Time per Image (ms)': 3.0164504051208496,
 'Number of parameters (mil)': 21.335972,
 'Pretrained Models': True,
 'Test accuracy (%)': 33.1,
 'Test loss (Cross Entropy)': 3.165004824615785,
 'Training Time (seconds)': 71.23479127883911,
 'approach': 'classical_ml',
 'backbone': 'resnet34',
 'classifier': 'logreg',
 'test_acc': 0.331,
 'val_acc': 0.3524}


### Logistic Regression with `resnet50`\Extract features using the `resnet50` architecture scaled to `32x32` and fit Logistic Regression.

In [11]:
_, eval_tf, _ = make_transforms(img_size=32)
results = classical_ml_experiment(
    ds=ds, 
    eval_tf=eval_tf, 
    device=device, 
    backbone="resnet50", 
    clf_type="logreg", 
    batch_size=256, 
    save_dir=save_dir, 
    img_size=32
)
print("Final Result:")
pprint.pprint(results)

[resnet50 | logreg @ 32] Extracting features ...


/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))
/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Saved -> experiments/classical_ml/classical_ml_logreg_resnet50_32.json
Final Result:
{'GPU': True,
 'Image Size': 32,
 'Inference Time per Image (ms)': 3.073228597640991,
 'Number of parameters (mil)': 23.712932,
 'Pretrained Models': True,
 'Test accuracy (%)': 33.12,
 'Test loss (Cross Entropy)': 8.242160500283273,
 'Training Time (seconds)': 80.81979894638062,
 'approach': 'classical_ml',
 'backbone': 'resnet50',
 'classifier': 'logreg',
 'test_acc': 0.3312,
 'val_acc': 0.3395}


### Logistic Regression with `efficientnet_b0`\Extract features using the `efficientnet_b0` architecture scaled to `32x32` and fit Logistic Regression.

In [12]:
_, eval_tf, _ = make_transforms(img_size=32)
results = classical_ml_experiment(
    ds=ds, 
    eval_tf=eval_tf, 
    device=device, 
    backbone="efficientnet_b0", 
    clf_type="logreg", 
    batch_size=256, 
    save_dir=save_dir, 
    img_size=32
)
print("Final Result:")
pprint.pprint(results)

[efficientnet_b0 | logreg @ 32] Extracting features ...


/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))
/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Saved -> experiments/classical_ml/classical_ml_logreg_efficientnet_b0_32.json
Final Result:
{'GPU': True,
 'Image Size': 32,
 'Inference Time per Image (ms)': 3.0331598281860352,
 'Number of parameters (mil)': 4.135648,
 'Pretrained Models': True,
 'Test accuracy (%)': 24.759999999999998,
 'Test loss (Cross Entropy)': 3.3279340358175142,
 'Training Time (seconds)': 46.73570990562439,
 'approach': 'classical_ml',
 'backbone': 'efficientnet_b0',
 'classifier': 'logreg',
 'test_acc': 0.2476,
 'val_acc': 0.2606}


### Logistic Regression with `efficientnet_b1`\Extract features using the `efficientnet_b1` architecture scaled to `32x32` and fit Logistic Regression.

In [13]:
_, eval_tf, _ = make_transforms(img_size=32)
results = classical_ml_experiment(
    ds=ds, 
    eval_tf=eval_tf, 
    device=device, 
    backbone="efficientnet_b1", 
    clf_type="logreg", 
    batch_size=256, 
    save_dir=save_dir, 
    img_size=32
)
print("Final Result:")
pprint.pprint(results)

[efficientnet_b1 | logreg @ 32] Extracting features ...


/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))
/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Saved -> experiments/classical_ml/classical_ml_logreg_efficientnet_b1_32.json
Final Result:
{'GPU': True,
 'Image Size': 32,
 'Inference Time per Image (ms)': 3.015800619125366,
 'Number of parameters (mil)': 6.641284,
 'Pretrained Models': True,
 'Test accuracy (%)': 20.76,
 'Test loss (Cross Entropy)': 3.549492881975026,
 'Training Time (seconds)': 227.88315796852112,
 'approach': 'classical_ml',
 'backbone': 'efficientnet_b1',
 'classifier': 'logreg',
 'test_acc': 0.2076,
 'val_acc': 0.2196}


/Users/david/working/antigravity/CS5242/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 5. Comprehensive Results Summary
Below we compile all generated JSON result files from `experiments/classical_ml` back into a single table for an easy comparison of validation accuracy, test accuracy, execution times, and more.

In [14]:
import pandas as pd
from IPython.display import display
import numpy as np

json_files = save_dir.glob("*.json")
data = []
for f in json_files:
    try:
        with open(f, "r") as file:
            data.append(json.load(file))
    except Exception as e:
        print(f"Skipping {f}: {e}")

if data:
    df = pd.DataFrame(data)
    df["NetScore"] = 20 * np.log10((df["Test accuracy (%)"] ** 2) / ((df["Inference Time per Image (ms)"] ** 0.5) * (df["Number of parameters (mil)"] ** 0.5)))
    
    cols = ["approach", "classifier", "backbone", "Image Size", "Number of parameters (mil)", 
            "val_acc", "Test accuracy (%)", "Test loss (Cross Entropy)", "Training Time (seconds)", "Inference Time per Image (ms)", "NetScore"]
    df_present = df[[c for c in cols if c in df.columns]]
    df_present = df_present.sort_values(by=["classifier", "backbone"]).reset_index(drop=True)
    
    if "Test accuracy (%)" in df_present.columns:
        df_present["Test accuracy (%)"] = df_present["Test accuracy (%)"].round(2).astype(str) + "%"
    if "val_acc" in df_present.columns:
        df_present["val_acc"] = (df_present["val_acc"] * 100).round(2).astype(str) + "%"
    if "Test loss (Cross Entropy)" in df_present.columns:
        df_present["Test loss (Cross Entropy)"] = df_present["Test loss (Cross Entropy)"].round(4)
    if "Training Time (seconds)" in df_present.columns:
        df_present["Training Time (seconds)"] = df_present["Training Time (seconds)"].round(2)
    if "Inference Time per Image (ms)" in df_present.columns:
        df_present["Inference Time per Image (ms)"] = df_present["Inference Time per Image (ms)"].round(2)
    if "NetScore" in df_present.columns:
        df_present["NetScore"] = df_present["NetScore"].round(2)
    
    display(df_present)
else:
    print("No JSON results found. Please run the cells above first to generate the outputs.")

,approach,classifier,backbone,Image Size,Number of parameters (mil),val_acc,Test accuracy (%),Test loss (Cross Entropy),Training Time (seconds),Inference Time per Image (ms),NetScore
0,classical_ml,linear_svm,convnext_tiny,224,27.897028,98.35%,93.9%,NaN,131.09,5.47,57.07
1,classical_ml,linear_svm,efficientnet_b0,224,4.135648,95.75%,91.94%,NaN,1751.99,3.95,66.41
2,classical_ml,linear_svm,efficientnet_b1,224,6.641284,96.65%,92.56%,NaN,346.10,4.57,63.84
3,classical_ml,linear_svm,resnet18,224,11.227812,86.98%,83.26%,NaN,267.09,3.43,60.96
4,classical_ml,linear_svm,resnet34,224,21.335972,91.35%,85.66%,NaN,186.98,3.76,58.26
5,classical_ml,linear_svm,resnet50,224,23.712932,97.1%,93.26%,NaN,673.69,5.05,58.01
6,classical_ml,logreg,convnext_tiny,32,27.897028,45.78%,44.18%,3.6852,22.75,3.02,46.55
7,classical_ml,logreg,efficientnet_b0,32,4.135648,26.06%,24.76%,3.3279,46.74,3.03,44.77
8,classical_ml,logreg,efficientnet_b1,32,6.641284,21.96%,20.76%,3.5495,227.88,3.02,39.67
9,classical_ml,logreg,resnet18,32,11.227812,34.1%,32.4%,3.1112,30.69,3.10,45.00
